Scenic Translation Algorithm

In [1]:
def construct_coach_behavior(action_json):
    function_lines = ["behavior coachBehavior():"]
    function_lines.append("    scene = simulation()")

    for action in action_json["actions"]:
        action_id = action["id"]
        args = action.get("args", {})
        
        if action_id == "moveTo":
            dest = args.get("dest", "")
            always = args.get("always", "")
            until = args.get("until", "")
            
            if dest and always and until:
                statement = f"    do {action_id}({dest}, {always}) until {until}"
            elif dest and until:
                statement = f"    do {action_id}({dest}) until {until}"
            elif dest:
                statement = f"    do {action_id}({dest})"
            else:
                statement = f"    do {action_id}()"
        
        elif action_id == "Idle":
            precondition = args.get("precondition", "")
            if precondition:
                statement = f"    do Idle() until {precondition}"
            else:
                statement = f"    do Idle()"

        elif action_id == "passTo":
            target = args.get("target", "")
            if target:
                statement = f"    do {action_id}({target})"
            else:
                statement = f"    do {action_id}()"

        else:
            statement = f"    do {action_id}()"
            s
        function_lines.append(statement)

    return "\n".join(function_lines)


In [2]:
example = {
    "actions": [
        {
            "id": "Idle",
            "args": {
                "precondition": "λ_precondition"
            },
            "constraints": {
                "λ_precondition": {
                    "logical": "A AND B",
                    "constraints": ["A", "B"],
                    "args": {
                        "A": {
                            "type": "InZone",
                            "args": {
                                "zone":  "C1"
                            }
                        },
                        "B": {
                            "type": "HasAngle",
                            "args": {
                                "ref": "player",
                                "r": 1.5
                            }
                        }
                    }
                }
            }
        },
        {
            "id": "moveTo",
            "args": {
                "dest": "λ_dest",
                "until": "λ_termination"
            },
            "constraints": {
                "λ_dest": {
                    "logical": "IF A THEN B ELSE (C AND D)",
                    "constraints": ["A", "B", "C", "D"],
                    "args": {
                        "A": {
                            "type": "IsVisible",
                            "args": {
                                "target": "goal"
                            }
                        },
                        "B": {
                            "type": "DistanceLessThan",
                            "args": {
                                "target": "goal",
                                "distance": 3.0
                            }
                        },
                        "C": {
                            "type": "InZone",
                            "args": {
                                "zone": "C3"
                            }
                        },
                        "D": {
                            "type": "HasAngle",
                            "args": {
                                "ref": "opponent",
                                "r": 2.0
                            }
                        }
                    }
                },
                "λ_termination": {
                    "logical": "(E AND F) OR (G AND H)",
                    "constraints": ["E", "F", "G", "H"],
                    "args": {
                        "E": {
                            "type": "InZone",
                            "args": {
                                "zone": "C4"
                            }
                        },
                        "F": {
                            "type": "IsVisible",
                            "args": {
                                "target": "teammate"
                            }
                        },
                        "G": {
                            "type": "DistanceGreaterThan",
                            "args": {
                                "target": "player",
                                "distance": 1.5
                            }
                        },
                        "H": {
                            "type": "HasAngle",
                            "args": {
                                "ref": "goal",
                                "r": 3.5
                            }
                        }
                    }
                }
            }
        },
        {
            "id": "Idle",
            "args": {
                "precondition": "λ_precondition"
            },
            "constraints": {
                "λ_precondition2": {
                    "logical": "C AND D",
                    "constraints": ["C", "D"],
                    "args": {
                        "C": {
                            "type": "InZone",
                            "args": {
                                "zone": "C5"
                            }
                        },
                        "D": {
                            "type": "DistanceLessThan",
                            "args": {
                                "target": "goal",
                                "distance": 2.0
                            }
                        }
                    }
                }
            }
        },
        {
            "id": "passTo",
            "args": {
                "target": "teammate"
            },
            "constraints": {
                "λ_dest": {
                    "logical": "A AND B",
                    "constraints": ["A", "B"],
                    "args": {
                        "A": {
                            "type": "IsVisible",
                            "args": {
                                "target": "teammate"
                            }
                        },
                        "B": {
                            "type": "DistanceLessThan",
                            "args": {
                                "target": "teammate",
                                "distance": 4.0
                            }
                        }
                    }
                }
            }
        }
    ]
}


In [ ]:
import re

def synthesize_conditionals(expression):
    expression = re.sub(r'\bIF\s+(.*?)\s+THEN\s+(.*?)\s+ELSE\s+(.*?)\b', r'(\2 if \1 else \3)', expression)
    expression = expression.replace("AND", "and").replace("OR", "or") 
    return expression

def get_args(action, constraint_name):
    args = action.get("constraints", {}).get(constraint_name, {}).get("args", {})
    formatted_args = []
    for arg_name, details in args.items():
        arg_type = details["type"]
        arg_values = ", ".join([f"'{key}': {repr(value)}" for key, value in details["args"].items()])
        formatted_args.append(f"{arg_name} = {arg_type}({{{arg_values}}})")
    return "\n".join(formatted_args)

def create_constraint_definitions(action_index, example):
    action = example["actions"][action_index]
    definitions = []
    for constraint_name in action.get("constraints", {}):
        print(action, constraint_name)
        constraint_def = get_args(action, constraint_name)
        definitions.append(constraint_def)
    return "\n".join(definitions)

def create_lambda_dest(action):
    constraints = action.get("constraints", {}).get("λ_dest", {})
    lambda_def = "def λ_dest(scene, sample):\n"
    logical_expr = constraints.get("logical", "")
    if not logical_expr:
        return lambda_def + "    return None  # No logical expression provided\n"

    logical_expr = synthesize_conditionals(logical_expr)
    for constraint_name in constraints.get("constraints", []):
        verify_call = f"{constraint_name}(scene, sample)"
        logical_expr = logical_expr.replace(constraint_name, verify_call)
    
    lambda_def += f"    return {logical_expr}\n"
    return lambda_def

def create_lambda_termination(action):
    constraints = action.get("constraints", {}).get("λ_termination", {})
    lambda_def = "def λ_termination(scene, sample):\n"
    logical_expr = constraints.get("logical", "")
    
    if not logical_expr:
        return lambda_def + "    return None  # No logical expression provided\n"

    logical_expr = synthesize_conditionals(logical_expr)
    for constraint_name in constraints.get("constraints", []):
        verify_call = f"{constraint_name}(scene, sample)"
        logical_expr = logical_expr.replace(constraint_name, verify_call)
    
    lambda_def += f"    return {logical_expr}\n"
    return lambda_def

def create_lambda_precondition(action):
    constraints = action.get("constraints", {}).get("λ_precondition", {})
    lambda_def = "def λ_precondition(scene, sample):\n"
    logical_expr = constraints.get("logical", "")
    
    if not logical_expr:
        return lambda_def + "    return None  # No logical expression provided\n"

    logical_expr = synthesize_conditionals(logical_expr)
    for constraint_name in constraints.get("constraints", []):
        verify_call = f"{constraint_name}(scene, sample)"
        logical_expr = logical_expr.replace(constraint_name, verify_call)
    
    lambda_def += f"    return {logical_expr}\n"
    return lambda_def


In [4]:
def generate_all_constraints_and_lambdas(example):
    for i, action in enumerate(example["actions"]):
        print(f"### Constraints and Lambda Functions for Action {i + 1}: {action['id']}")
        
        constraint_definitions = create_constraint_definitions(i, example)
        if constraint_definitions:
            print("Constraint Definitions:")
            print(constraint_definitions)
        else:
            print("No Constraint Definitions.")

        lambda_precondition_code = create_lambda_precondition(action)
        if lambda_precondition_code:
            print("\nλ_precondition Function:")
            print(lambda_precondition_code) 

        lambda_dest_code = create_lambda_dest(action)
        if lambda_dest_code:
            print("\nλ_dest Function:")
            print(lambda_dest_code)

        lambda_termination_code = create_lambda_termination(action)
        if lambda_termination_code:
            print("\nλ_termination Function:")
            print(lambda_termination_code)

        print("\n" + "=" * 40 + "\n")

generate_all_constraints_and_lambdas(example)



### Constraints and Lambda Functions for Action 1: Idle
Constraint Definitions:
A = InZone({'zone': 'C1'})
B = HasAngle({'ref': 'player', 'r': 1.5})

λ_precondition Function:
def λ_precondition(scene, sample):
    return A(scene, sample) and B(scene, sample)


λ_dest Function:
def λ_dest(scene, sample):
    return None  # No logical expression provided


λ_termination Function:
def λ_termination(scene, sample):
    return None  # No logical expression provided



### Constraints and Lambda Functions for Action 2: moveTo
Constraint Definitions:
A = IsVisible({'target': 'goal'})
B = DistanceLessThan({'target': 'goal', 'distance': 3.0})
C = InZone({'zone': 'C3'})
D = HasAngle({'ref': 'opponent', 'r': 2.0})
E = InZone({'zone': 'C4'})
F = IsVisible({'target': 'teammate'})
G = DistanceGreaterThan({'target': 'player', 'distance': 1.5})
H = HasAngle({'ref': 'goal', 'r': 3.5})

λ_precondition Function:
def λ_precondition(scene, sample):
    return None  # No logical expression provided


λ_dest

In [5]:
class InZone:
    def __init__(self, args):
        self.args = args

class HasAngle:
    def __init__(self, args):
        self.args = args

class IsVisible:
    def __init__(self, args):
        self.args = args

class DistanceLessThan:
    def __init__(self, args):
        self.args = args

class DistanceGreaterThan:
    def __init__(self, args):
        self.args = args


In [6]:
# constraints = constraint_definitions.split('\n')
# keys = {}

# for i, constraint in enumerate(constraints):
#     if constraint.strip(): 
#         try:
#             exec(constraint, globals(), keys)
#         except Exception as e:
#             print(f"Error processing constraint '{constraint}': {e}")

# print(keys)

# list_of_keys = list((keys.keys()))

# first_key = list_of_keys[0]
# first_object = keys[first_key]

# print(f"Arguments of the first object ({first_key}): {first_object.args}")

"""not using anymore"""


'not using anymore'

In [7]:
behavior_code = construct_coach_behavior(example)
print(behavior_code)

behavior coachBehavior():
    scene = simulation()
    do Idle() until λ_precondition
    do moveTo(λ_dest) until λ_termination
    do Idle() until λ_precondition
    do passTo(teammate)
